# TrialSieve: benchmarks and final evaluation
Settings: **GPU T4 x2**, Internet **On**, secret `HF_TOKEN`. Run sections 0-4 top to bottom; section 5 only when told.

In [ ]:
!rm -rf /kaggle/working/trialsieve && git clone -q https://github.com/ibrahimmlk/trialsieve.git /kaggle/working/trialsieve
%cd /kaggle/working/trialsieve
!git log --oneline -1
!pip install -q -U peft pytest
!pip uninstall -y -q torchao
import os; os.environ['CUDA_VISIBLE_DEVICES'] = '0'
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi, snapshot_download
login(UserSecretsClient().get_secret('HF_TOKEN'))
api = HfApi()
REPO, DRAFT_REPO = 'ibrahimmlk/trialsieve-qwen2.5-1.5b-lora', 'ibrahimmlk/trialsieve-qwen2.5-0.5b-draft-lora'
snapshot_download(REPO, local_dir='outputs/lora-qwen1.5b/final', allow_patterns=['adapter_*', '*.json', '*.jinja'])

## 0. Unit tests (attention, manual model vs HF, KV cache, speculative)

In [ ]:
!python -m pytest -q tests

## 1. Re-evaluate validation with pure greedy decoding (repetition penalty off)

In [ ]:
!python scripts/evaluate.py --split val --tag base
!python scripts/evaluate.py --split val --adapter outputs/lora-qwen1.5b/final --tag finetuned

## 2. Fine-tune the 0.5B draft model on the same train split (for speculative decoding)

In [ ]:
!python scripts/train.py --model Qwen/Qwen2.5-0.5B-Instruct --output-dir outputs/lora-qwen0.5b-draft
api.create_repo(DRAFT_REPO, exist_ok=True)
api.upload_folder(repo_id=DRAFT_REPO, folder_path='outputs/lora-qwen0.5b-draft/final')
for f in ['log_history.json', 'train_config.json']:
    api.upload_file(repo_id=DRAFT_REPO, path_or_fileobj=f'outputs/lora-qwen0.5b-draft/{f}', path_in_repo=f'training/{f}')

## 3. GPU benchmarks: KV cache, attention kernels, speculative decoding

In [ ]:
!python scripts/benchmark.py --part all --adapter outputs/lora-qwen1.5b/final --draft-adapter outputs/lora-qwen0.5b-draft/final

## 4. Upload results

In [ ]:
api.upload_folder(repo_id=REPO, folder_path='results', path_in_repo='results')
print('done')

## 5. FINAL held-out test evaluation (run once, only when told)

In [ ]:
!python scripts/evaluate.py --split test --final --tag base
!python scripts/evaluate.py --split test --final --adapter outputs/lora-qwen1.5b/final --tag finetuned
api.upload_folder(repo_id=REPO, folder_path='results', path_in_repo='results')